# Notebook 3 — Non-CMF Unlearning Baselines · CIFAR-100 (Paper Protocol)

**Paper:** *An Illusion of Unlearning?* (Gao et al., AISTATS 2026 · arXiv:2604.08271v1)

**Methods:** NegGrad+, Random-label, SalUn, SCRUB, UNSIR, SVD

**Protocol — matches paper exactly (Table 1 / Appendix A.3):**
- Forget set = **one entire class** (all ~500 training images of that class).
- Retain set = all training images from the remaining 99 classes.
- 5 representative CIFAR-100 classes `{0},{1},{2},{3},{5}` as forget class (§A.3).
- Results **averaged** (mean ± std) across the 5 forget classes.
- **Output accuracy** evaluated on the **held-out test set** (100 test samples per class).
- **Probe & NCC** features built from the full 50 000-sample training set D = D_r ∪ D_f;
  evaluated on test-set forget / retain subsets (paper §3.2 / eq. 3).
- Hyperparameters: CIFAR-100 / ResNet-18 column of Table 4.

**Outputs:** `{method}_class{c}_seed{seed}.pt` per run; `results_nocmf_cifar100_paper.csv` summary.

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn pytorch-lightning torchmetrics')

In [ ]:
import os, sys, json, random, math, time, collections
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib; import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})
print('PyTorch:', torch.__version__, '  CUDA:', torch.cuda.is_available())

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} remote set-url origin https://github.com/tiensinh2/CMF_Unlearning.git')
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
result = subprocess.run(['git','-C',REPO_DIR,'rev-parse','HEAD'],
                        capture_output=True, text=True)
REPO_COMMIT = result.stdout.strip() or 'main'
print('Repo commit:', REPO_COMMIT)

In [ ]:
# ══ SET THIS to the Kaggle dataset mount path containing theta_o checkpoint from Notebook 1 ══
CKPT_DATASET_DIR = '/kaggle/input/datasets/kiethe/cmf-nb1-c100'

SPLIT_CFG = os.path.join(CKPT_DATASET_DIR, 'split_config_cifar100.json')
if os.path.exists(SPLIT_CFG):
    with open(SPLIT_CFG) as f: NB1_CFG = json.load(f)
    DATASET     = NB1_CFG.get('dataset',     'cifar100')
    NUM_CLASSES = NB1_CFG.get('num_classes', 100)
else:
    DATASET     = 'cifar100'
    NUM_CLASSES = 100

ARCH      = 'resnet18'
TEST_MODE = False

# Paper §A.3: CIFAR-100 single-class forget — 5 representative classes
FORGET_CLASSES = [0, 1, 2, 3, 5]
SEEDS          = [0]
THETA_O_SEED   = 0

CKPT_ROOT = '/kaggle/working/checkpoints/unlearn_nocmf_cifar100_paper'
os.makedirs(CKPT_ROOT, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'DATASET={DATASET}  ARCH={ARCH}  NUM_CLASSES={NUM_CLASSES}  device={device}')
print(f'Forget classes to sweep: {FORGET_CLASSES}')

In [ ]:
import torchvision, torchvision.transforms as transforms

# CIFAR-100 standard normalization statistics
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD  = (0.2675, 0.2565, 0.2761)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

full_train      = torchvision.datasets.CIFAR100('/kaggle/working/data', train=True,
                                                download=True,  transform=transform_train)
full_train_eval = torchvision.datasets.CIFAR100('/kaggle/working/data', train=True,
                                                download=False, transform=transform_test)
test_set = torchvision.datasets.CIFAR100('/kaggle/working/data', train=False,
                                         download=True, transform=transform_test)

test_targets = torch.tensor(test_set.targets)
TEST_CLASS_IDX = {
    c: (test_targets == c).nonzero(as_tuple=True)[0].tolist()
    for c in range(NUM_CLASSES)
}
train_targets = torch.tensor(full_train.targets)
TRAIN_CLASS_IDX = {
    c: (train_targets == c).nonzero(as_tuple=True)[0].tolist()
    for c in range(NUM_CLASSES)
}
full_train_loader = torch.utils.data.DataLoader(
    full_train, batch_size=256, shuffle=False, num_workers=2)

print(f'Train: {len(full_train)}  Test: {len(test_set)}')
print(f'Test samples per class: {len(TEST_CLASS_IDX[0])} (expected 100 for CIFAR-100)')

In [ ]:
from models.resnet import ResNet18

def build_model():
    return ResNet18(num_classes=NUM_CLASSES, dataset=DATASET).to(device)


@torch.no_grad()
def extract_features_resnet(model, loader):
    model.eval()
    feats, labs = [], []
    for x, y in loader:
        x = x.to(device)
        buf = []
        hook = model.avgpool.register_forward_hook(
            lambda m, i, o: buf.append(o.flatten(1).detach().cpu())
        )
        model(x)
        hook.remove()
        feats.append(buf[0]); labs.append(y)
    return torch.cat(feats), torch.cat(labs)


@torch.no_grad()
def eval_acc(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        correct += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / max(total, 1)


def run_linear_probe(model, train_retain_ldr, train_forget_ldr,
                     test_retain_ldr, test_forget_ldr,
                     n_epochs=50, lr=1e-2):
    """
    Paper §3.2: probe is trained on frozen features of ALL training data D = D_r ∪ D_f,
    then evaluated separately on test-set retain and test-set forget subsets.
    """
    Xtr, ytr = extract_features_resnet(model, train_retain_ldr)
    Xfg, yfg = extract_features_resnet(model, train_forget_ldr)
    Xall = torch.cat([Xtr, Xfg])
    yall = torch.cat([ytr, yfg])

    head = nn.Linear(Xall.size(1), NUM_CLASSES).to(device)
    opt  = optim.SGD(head.parameters(), lr=lr, momentum=0.9)
    ldr  = torch.utils.data.DataLoader(
               torch.utils.data.TensorDataset(Xall, yall), batch_size=256, shuffle=True)
    for _ in range(n_epochs):
        head.train()
        for bx, by in ldr:
            opt.zero_grad()
            F.cross_entropy(head(bx.to(device)), by.to(device)).backward()
            opt.step()
    head.eval()

    with torch.no_grad():
        Xte_r, yte_r = extract_features_resnet(model, test_retain_ldr)
        Xte_f, yte_f = extract_features_resnet(model, test_forget_ldr)
        ret_acc = (head(Xte_r.to(device)).argmax(1).cpu() == yte_r).float().mean().item() * 100
        fgt_acc = (head(Xte_f.to(device)).argmax(1).cpu() == yte_f).float().mean().item() * 100
    return ret_acc, fgt_acc


def run_ncc(model, train_retain_ldr, train_forget_ldr,
            test_retain_ldr, test_forget_ldr):
    """
    Paper eq. 3: class means μ_k computed from ALL training samples (D_r ∪ D_f).
    NCC accuracy evaluated on test-set retain and test-set forget subsets.
    """
    Xtr, ytr = extract_features_resnet(model, train_retain_ldr)
    Xfg, yfg = extract_features_resnet(model, train_forget_ldr)
    Xall = torch.cat([Xtr, Xfg])
    yall = torch.cat([ytr, yfg])
    means = []
    for c in range(NUM_CLASSES):
        mask = (yall == c)
        mu = Xall[mask].mean(0) if mask.any() else torch.zeros(Xall.size(1))
        means.append(mu)
    M = torch.stack(means)

    Xte_r, yte_r = extract_features_resnet(model, test_retain_ldr)
    Xte_f, yte_f = extract_features_resnet(model, test_forget_ldr)
    ret_pred = torch.cdist(Xte_r.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(dim=1)
    fgt_pred = torch.cdist(Xte_f.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(dim=1)
    ret_acc  = (ret_pred == yte_r).float().mean().item() * 100
    fgt_acc  = (fgt_pred == yte_f).float().mean().item() * 100
    return ret_acc, fgt_acc


def eval_three_metrics(model,
                       test_retain_ldr, test_forget_ldr,
                       train_retain_ldr, train_forget_ldr):
    """
    Three-metric evaluation matching paper Table 1 exactly.

    Output  — direct model accuracy on HELD-OUT TEST-SET retain / forget images.
    Probe   — linear head trained on ALL TRAINING features (D_r ∪ D_f),
              evaluated on test-set forget / retain.
    NCC     — class means from ALL TRAINING features, evaluated on test-set.
    """
    out_ret = eval_acc(model, test_retain_ldr)
    out_fgt = eval_acc(model, test_forget_ldr)
    lp_ret, lp_fgt = run_linear_probe(
        model,
        train_retain_ldr, train_forget_ldr,
        test_retain_ldr,  test_forget_ldr
    )

    ncc_ret, ncc_fgt = run_ncc(
        model,
        train_retain_ldr, train_forget_ldr,
        test_retain_ldr,  test_forget_ldr
    )

    return {
        'output_retain_acc': out_ret, 'output_forget_acc': out_fgt,
        'probe_retain_acc':  lp_ret,  'probe_forget_acc':  lp_fgt,
        'ncc_retain_acc':    ncc_ret, 'ncc_forget_acc':    ncc_fgt,
    }

print('Eval helpers ready.')

In [ ]:
# Hyperparameters — CIFAR-100 / ResNet-18 rows of Table 4 (arXiv:2604.08271v1).
HPARAM_SOURCE = 'table4'
METHODS = [
    # (method_key, dispatch_key, epochs, lr, batch_size)
    ('random_label',        'random_label',        3,  3e-3,  128),  # Table 4 line 2333
    ('salun',               'salun',               3,  1e-3,  128),  # Table 4 line 2342, threshold=0.5
    ('grad_ascent_descent', 'grad_ascent_descent', 3,  5e-3,  128),  # Table 4 line 2351, grad-clip=1.0
    ('scrub',               'scrub',               3,  1e-3,   64),  # Table 4 line 2360, sgda-bsz=64, msteps=2
    ('tarun',               'tarun',               3,  3e-5,  128),  # Table 4 line 2369, 3 epochs impair/repair
    ('SVD',                 'SVD',                 1,  0.0,   990),  # Table 4 line 2380, alpha_r=1000, alpha_f=30
]


def make_unlearn_args(method_key, lr, epochs, batch_size, forget_class, retain_idx, seed=0):
    """
    Build args namespace for one whole-class unlearning run.
    forget_class : int  — the single class index being forgotten
    retain_idx   : list — training-set indices of all retain samples
    """
    import argparse
    forget_idx = TRAIN_CLASS_IDX[forget_class]
    return argparse.Namespace(
        dataset=DATASET, arch=ARCH, num_classes=NUM_CLASSES,
        class_label_names=list(range(NUM_CLASSES)),
        unlearn_method=method_key,
        unlearn_class=[forget_class],
        batch_size=batch_size, test_batch_size=256,
        lr=lr, momentum=0.9, weight_decay=5e-4,
        epochs_or_steps=epochs,
        seed=seed,
        num_retain_samples=len(retain_idx),
        num_forget_samples=len(forget_idx),
        grad_norm_clip=(1.0 if 'grad_ascent' in method_key else None),
        # SVD params — CIFAR-100: alpha_r=1000, alpha_f=30, samples=990 (Table 4 line 2380)
        SVD_alpha_r=1000, SVD_alpha_f=30, SVD_samples=990, SVD_max_patches=10000,
        freeze_except_last=False,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=epochs,
        salun_threshold=0.5,
        # UNSIR: impair_lr=3e-5 (Table 4 line 2369), same as main lr
        tarun_impair_lr=3e-5, tarun_samples_per_class=1000,
        dry_run=False, no_cuda=False, no_mps=True, gamma=0.5,
        data_path='/kaggle/working/data',
        remove_FC=False, CMFClassifier=False,
        prob_batch_size=256, lp_every=0, ncc_every=0,
        repo_commit=REPO_COMMIT, test_mode=TEST_MODE,
    )

print('Method helpers ready.')

In [ ]:
from unlearn import unlear_func

# Load theta_o checkpoint from Notebook 1 (CIFAR-100)
theta_o_path = os.path.join(CKPT_DATASET_DIR, 'original_resnet18_cifar100.pt')
assert os.path.exists(theta_o_path), (
    f'Missing theta_o: {theta_o_path}\n'
    f'Run Notebook 1 (CIFAR-100) first and set CKPT_DATASET_DIR correctly.'
)

all_results = []

for forget_class in FORGET_CLASSES:
    for seed in SEEDS:

        # ── Build whole-class forget / retain splits from training set ──────────
        # Paper §A.3: forget set = all samples of one class (~500 for CIFAR-100).
        forget_train_idx = TRAIN_CLASS_IDX[forget_class]           # ~500 samples
        retain_train_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TRAIN_CLASS_IDX[c]
        ]                                                            # ~49 500 samples

        retain_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, retain_train_idx),
            batch_size=128, shuffle=True, num_workers=2)
        forget_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, forget_train_idx),
            batch_size=128, shuffle=True, num_workers=2)

        train_retain_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, retain_train_idx),
            batch_size=256, shuffle=False, num_workers=2)
        train_forget_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, forget_train_idx),
            batch_size=256, shuffle=False, num_workers=2)

        # ── Test-set loaders (100 test samples per class for CIFAR-100) ──────────
        test_forget_idx = TEST_CLASS_IDX[forget_class]             # 100 test samples
        test_retain_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TEST_CLASS_IDX[c]
        ]                                                            # 9 900 test samples
        test_forget_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_forget_idx),
            batch_size=256, shuffle=False, num_workers=2)
        test_retain_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_retain_idx),
            batch_size=256, shuffle=False, num_workers=2)

        # ── Inner loop: one run per method ──────────────────────────────────────
        for method_key, dispatch_key, epochs, lr, batch_sz in METHODS:
            tag = f'{method_key}_class{forget_class}_seed{seed}'
            ckpt_path = f'{CKPT_ROOT}/{tag}.pt'

            if os.path.exists(ckpt_path):
                print(f'[{tag}] exists — skipping.')
                ck = torch.load(ckpt_path, map_location=device)
                all_results.append(ck['metrics'])
                continue

            print(f'\n[{tag}] forget_class={forget_class}  method={method_key}'
                  f'  lr={lr}  epochs={epochs}  batch={batch_sz}')
            torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
            model = build_model()
            ck_o  = torch.load(theta_o_path, map_location=device)
            model.load_state_dict(
                ck_o['model_state_dict'] if 'model_state_dict' in ck_o else ck_o)

            args = make_unlearn_args(
                method_key, lr, epochs, batch_sz,
                forget_class, retain_train_idx, seed=seed
            )
            fn = unlear_func[dispatch_key]

            t0 = time.time()
            try:
                model = fn(
                    args=args, model=model, device=device,
                    retain_loader=retain_loader,
                    forget_loader=forget_loader,
                    train_loader=full_train_loader,
                    test_loader=torch.utils.data.DataLoader(
                        test_set, batch_size=256, shuffle=False, num_workers=2),
                    optimizer=None, epochs=epochs,
                    test_forget_loader=test_forget_ldr,
                    train_dataset=full_train,
                    val_index=retain_train_idx,
                )
            except Exception as e:
                import traceback; traceback.print_exc()
                print(f'  ERROR in {tag}: {e}')
                continue
            wall_min = (time.time() - t0) / 60

            metrics = eval_three_metrics(
                model,
                test_retain_ldr, test_forget_ldr,
                train_retain_eval_ldr, train_forget_eval_ldr
            )
            metrics.update({
                'method': method_key, 'forget_class': forget_class, 'seed': seed,
                'lr': lr, 'epochs': epochs, 'batch_size': batch_sz,
                'wall_clock_minutes': wall_min,
                'hparam_source': HPARAM_SOURCE,
                'n_forget_train': len(forget_train_idx),
                'n_retain_train': len(retain_train_idx),
            })

            torch.save({
                'model_state_dict': model.state_dict(),
                'config': {
                    'method': method_key, 'dataset': DATASET, 'arch': ARCH,
                    'forget_class': forget_class, 'seed': seed,
                    'lr': lr, 'epochs': epochs, 'num_classes': NUM_CLASSES,
                    'protocol': 'whole_class_single',
                    'hparam_source': HPARAM_SOURCE,
                    'repo_commit': REPO_COMMIT, 'test_mode': TEST_MODE,
                    'paper': 'arXiv:2604.08271v1 Table 4 CIFAR-100',
                },
                'seed': seed,
                'metrics': metrics,
            }, ckpt_path)
            print(f'  Saved {ckpt_path}')
            print(f'  output  R={metrics["output_retain_acc"]:.2f}%  '
                  f'F={metrics["output_forget_acc"]:.2f}%')
            print(f'  probe   R={metrics["probe_retain_acc"]:.2f}%  '
                  f'F={metrics["probe_forget_acc"]:.2f}%')
            print(f'  ncc     R={metrics["ncc_retain_acc"]:.2f}%  '
                  f'F={metrics["ncc_forget_acc"]:.2f}%')
            all_results.append(metrics)


# ── Aggregate: mean ± std over all forget classes (matches paper Table 1) ──────
df = pd.DataFrame(all_results)
csv_path = f'{CKPT_ROOT}/results_nocmf_cifar100_paper.csv'
df.to_csv(csv_path, index=False)
print(f'\nAll done. Per-run results saved: {csv_path}')

if df.empty:
    print('WARNING: no successful runs — all methods errored. Check errors above.')
else:
    metric_cols = [
        'output_retain_acc', 'output_forget_acc',
        'probe_retain_acc',  'probe_forget_acc',
        'ncc_retain_acc',    'ncc_forget_acc',
    ]
    summary = (
        df.groupby('method')[metric_cols]
        .agg(['mean', 'std'])
        .round(2)
    )
    summary.columns = ['_'.join(c) for c in summary.columns]
    summary_path = f'{CKPT_ROOT}/results_nocmf_cifar100_paper_summary.csv'
    summary.to_csv(summary_path)
    print(f'Summary (mean±std over {len(FORGET_CLASSES)} forget classes) saved: {summary_path}')
    print('\n=== Mean across all forget classes ===')
    mean_cols = [c for c in summary.columns if c.endswith('_mean')]
    print(summary[mean_cols].to_string())